# M49 — Zero-shot cross-dataset transfer: ICBHI 2017 → SPRSound 2022

**Model ID:** `M49` · **Chunk:** F (generalization) · **Contributor:** OWMTL team
**Ablation group:** `ood_generalization` · **Training:** none — inference only

## What this answers

Dr. Khan's review comment: *"Test your best trained model with another dataset, i.e., SPRSound
(2022, open access) or HF Lung or the CirCor DigiScope."*

The best model is **MobileNetV2 + SpecAugment (M22_v2)**, ICBHI challenge score **0.5602**
on the corrected 60/40 patient-independent partition (2,636 cycles, 47 patients). This notebook
takes that checkpoint **frozen**, changes nothing about it, and scores it on SPRSound — a
paediatric corpus recorded with a different electronic stethoscope at a different sample rate.

Nothing is trained or fine-tuned here. Every weight is the one that produced 0.5602.

## Why SPRSound and not the other two

| Dataset | Sound type | Labels | Verdict |
|---|---|---|---|
| **SPRSound 2022** | lung | event-level, 7 adventitious classes | **primary** — maps onto our 4 classes, open on GitHub, no registration |
| HF Lung V1 | lung | inhale/exhale/CAS/DAS segments | possible, but access needs a request form and the labels are segmentation, not cycle classes |
| CirCor DigiScope | **heart (PCG)** | murmur present/absent | not a respiratory transfer test at all — see `M50` for the negative-control framing |

CirCor is a phonocardiogram corpus. A crackle/wheeze classifier has no label overlap with it, so
it cannot measure generalization. It is still useful, but as a *negative control* (does the model
confidently hallucinate lung pathology in heart audio?), which is a separate notebook.

## What gets measured

1. **Sanity gate** — the checkpoint is re-scored on the ICBHI corrected test split and must
   reproduce 0.5602. A transfer number from an unverified checkpoint is worthless.
2. **Zero-shot 4-class** transfer under two label mappings (strict and broad).
3. **Zero-shot binary** Normal vs adventitious — the mapping that is semantically safe across corpora.
4. **Reference baselines on SPRSound** — always-Normal, prior-matched random, uniform.
   Without these the transfer score cannot be read.
5. **Prior-shift correction** — how much of the drop is class-prior shift rather than a broken
   representation.
6. **Calibration under shift** — ECE and confidence on SPRSound against the same on ICBHI.
7. **Age-stratified breakdown** — continuity with the Gap7 finding that adult-tuned acoustic
   priors degrade on children.

## Prerequisite — read this before running

You need `best_model.pth` from the **corrected** M22_v2 run. That file is **not in the repo**
(only the leaky-split variant survived in `Archive_Files (v4)/`). Produce it first with
`Asif's/M22_v2/m22-official-notebook.ipynb` using `VARIANT = "augmented"` and
`SPLIT_MODE = "corrected"`, then put the resulting `best_model.pth` where Cell 6 can find it.
Cell 6 refuses to run without it — it does not substitute a different checkpoint.

## Section 1 — Environment setup & dependencies

In [ ]:
# ============================================================
# CELL 1 — ENVIRONMENT
# ============================================================
import os, sys, json, glob, math, time, random, subprocess, tempfile, shutil, warnings
warnings.filterwarnings("ignore")

def _pip(pkg, imp=None):
    try:
        __import__(imp or pkg.split("==")[0].replace("-", "_"))
    except ImportError:
        print(f"installing {pkg} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

_pip("librosa")
_pip("soundfile")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
import librosa
import matplotlib
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (confusion_matrix, accuracy_score,
                             precision_recall_fscore_support)
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

if os.path.exists("/kaggle/working"):
    PLATFORM = "Kaggle"
elif "google.colab" in sys.modules or os.path.exists("/content"):
    PLATFORM = "Colab"
else:
    PLATFORM = "Local"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"platform : {PLATFORM}")
print(f"device   : {DEVICE}"
      + (f"  ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else ""))
print(f"torch    : {torch.__version__} | torchvision {torchvision.__version__}")
print(f"librosa  : {librosa.__version__}")

## Section 2 — Configuration

The audio front-end below is **copied verbatim from M22_v2**. Do not adjust any of it. The
checkpoint was fine-tuned on exactly this scaling (`power_to_db(ref=max)` then per-sample min-max
to [0,1], short cycles wrap-padded by repetition), and changing a single parameter would make the
transfer number measure our preprocessing rather than the model.

SPRSound is distributed at 8 kHz. `librosa.load(sr=16000)` resamples it up. That is safe here
because the mel filterbank stops at `f_max = 2000 Hz`, well under the 4 kHz Nyquist of the source,
so no band is fabricated — the resample only re-grids the frames. It is recorded and reported
either way.

In [ ]:
# ============================================================
# CELL 2 — CONFIG
# ============================================================
WORK = {"Kaggle": "/kaggle/working", "Colab": "/content", "Local": "."}[PLATFORM]

CFG = {
    # -- audio front-end: VERBATIM from M22_v2 / protocol section 2. Do not touch. ----
    "sample_rate": 16000, "duration_s": 8.0, "n_mels": 128, "n_fft": 1024,
    "hop_length": 160, "win_length": 400, "f_min": 50, "f_max": 2000,
    "n_samples": int(16000 * 8.0),
    "n_frames": None,                      # -> 801, computed below

    # -- model: must match the checkpoint exactly --------------------------------
    "arch": "mobilenet_v2", "pretrained": True, "freeze_features": False,
    "dropout": 0.3, "imagenet_normalize": True,
    "classes": ["Normal", "Crackle", "Wheeze", "Both"], "num_classes": 4,

    # -- inference ---------------------------------------------------------------
    "batch_size": 32, "num_workers": 2, "seed": SEED,
    "use_specaugment": False,              # inference: augmentation is training-time only

    # -- identity ----------------------------------------------------------------
    "model_id": "M49",
    "model_name": "Zero-shot cross-dataset transfer, M22_v2 -> SPRSound 2022",
    "source_model_id": "M22_v2",
    "source_score_icbhi": 0.5602,          # the number the sanity gate must reproduce
    "source_test_cycles": 2636,
    "source_test_patients": 47,
    "contributor": "OWMTL team",

    # -- statistics (Essential #10) ----------------------------------------------
    "n_bootstrap": 1000,

    # -- paths --------------------------------------------------------------------
    "out_dir":   f"{WORK}/results_M49",
    "cache_dir": f"{WORK}/cache_M49",
}
CFG["n_frames"] = 1 + math.floor(CFG["n_samples"] / CFG["hop_length"])   # 801
for d in (CFG["out_dir"], CFG["cache_dir"]):
    os.makedirs(d, exist_ok=True)

# ---------------------------------------------------------------------------
# SWITCHES — the only things you should normally change
# ---------------------------------------------------------------------------
REQUIRE_ICBHI_VERIFICATION = True   # False = skip the sanity gate. The transfer number is
                                    # then marked unverified in the results JSON. Only do this
                                    # if the ICBHI audio genuinely cannot be attached.

SPRSOUND_GIT = "https://github.com/SJTU-YONGFU-RESEARCH-GRP/SPRSound.git"
SPRSOUND_LOCAL = None               # set to a folder you already have, to skip the clone


print("=" * 70)
print(f"  {CFG['model_id']} — {CFG['model_name']}")
print("=" * 70)
for k in ("sample_rate", "n_mels", "n_frames", "f_min", "f_max", "arch",
          "source_model_id", "source_score_icbhi"):
    print(f"  {k:<22} {CFG[k]}")
print(f"  {'output':<22} {CFG['out_dir']}")

In [ ]:
# ============================================================
# OUTPUT SETUP — Google Drive, checked not assumed   [OWMTL_DRIVE_OUTPUT_V2]
# ============================================================
# Spliced from Asif's/engine/drive_setup.py -- do not hand-edit. Re-run
# `python3 "Asif's/engine/patch_colab_v2.py"` to refresh.
#
# On Colab this prompts for Drive access the first time. Accept it: without Drive every
# checkpoint and result lives in /content and disappears when the runtime disconnects, which
# on a 40-epoch run means losing about two hours.
#
# The old block in these notebooks did:
#     try:  drive.mount(...); DRIVE_MOUNTED = True
#     except Exception as e:  print("Falling back to ephemeral /content storage.")
# Two silent failures came out of that: re-mounting an already-mounted Drive raises and the
# except swallowed it, and a mounted-but-unwritable Drive passes makedirs() and only fails
# hours into training. This version PROBES that the directory is genuinely writable and
# RAISES if not. It never silently downgrades.
#
# RUN_MODE controls what happens to an existing OWMTL/<id> folder:
#   "auto"        completed run -> new version (<id>_v2, _v3, ...), old results preserved
#                 interrupted run (checkpoints, no results) -> reuse it so auto-resume works
#   "new_version" always a fresh versioned folder
#   "reuse"       use the folder as-is
#   "overwrite"   delete its contents first  (destructive)
import os, sys, shutil

RUN_MODE = "auto"     # <-- "reuse" to resume an interrupted run in place

import os
import shutil

__all__ = ["setup_output_dir", "mount_drive_if_available"]


def mount_drive_if_available(verbose=True):
    """Return (mounted: bool, detail: str). Never raises."""
    in_colab = "google.colab" in __import__("sys").modules or os.path.exists("/content")
    if not in_colab:
        return False, "not a Colab runtime"

    # Already mounted? Do NOT call mount() again -- that is what raises
    # "Mountpoint must not already contain files".
    if os.path.isdir("/content/drive/MyDrive"):
        if verbose:
            print("Google Drive: already mounted at /content/drive")
        return True, "already mounted"

    try:
        from google.colab import drive
    except Exception as e:
        return False, f"google.colab unavailable ({e})"

    try:
        drive.mount("/content/drive")
        if verbose:
            print("Google Drive: mounted at /content/drive")
        return True, "mounted"
    except Exception as e1:
        # Most common cause: a stale, non-empty mountpoint. force_remount clears it.
        try:
            drive.mount("/content/drive", force_remount=True)
            if verbose:
                print(f"Google Drive: force-remounted (first attempt failed: {e1})")
            return True, "force-remounted"
        except Exception as e2:
            return False, f"mount failed: {e1} | force_remount failed: {e2}"


def _probe_writable(path):
    """Actually write, read back and delete a file. Returns (ok, error)."""
    try:
        os.makedirs(path, exist_ok=True)
        probe = os.path.join(path, ".owmtl_write_probe")
        with open(probe, "w") as f:
            f.write("ok")
        with open(probe) as f:
            if f.read() != "ok":
                return False, "read-back mismatch"
        os.remove(probe)
        return True, None
    except Exception as e:
        return False, str(e)


def _classify(path):
    """'absent' | 'empty' | 'completed' | 'in_progress'."""
    if not os.path.isdir(path):
        return "absent"
    entries = [e for e in os.listdir(path) if not e.startswith(".")]
    if not entries:
        return "empty"
    for dirpath, _d, files in os.walk(path):
        for fn in files:
            if fn.startswith("results_") and fn.endswith(".json"):
                return "completed"
    for dirpath, _d, files in os.walk(path):
        for fn in files:
            if fn.endswith(".pth"):
                return "in_progress"
    return "in_progress"


def _next_version(root, name):
    """name -> name_v2 -> name_v3 ... first path that is absent or empty."""
    v = 2
    while True:
        cand = os.path.join(root, f"{name}_v{v}")
        if _classify(cand) in ("absent", "empty"):
            return cand
        v += 1
        if v > 99:
            raise RuntimeError(f"Refusing to version past {name}_v99 -- clean up {root}.")


def setup_output_dir(model_id, base_name="OWMTL", mode="auto", verbose=True):
    """Resolve a GUARANTEED-WRITABLE output directory.

    Returns (base_dir, info). Raises RuntimeError if nothing writable can be found -- never
    silently downgrades to ephemeral storage.
    """
    if mode not in ("auto", "new_version", "reuse", "overwrite"):
        raise ValueError(f"unknown mode {mode!r}")

    import sys
    in_colab = "google.colab" in sys.modules or os.path.exists("/content")
    mounted, detail = mount_drive_if_available(verbose=verbose) if in_colab else (False, "n/a")

    # Candidate roots, best first. Each is probed before use.
    if in_colab and mounted:
        roots = [(f"/content/drive/MyDrive/{base_name}", "Google Drive (persistent)"),
                 (f"/content/{base_name}", "Colab local (EPHEMERAL)")]
    elif in_colab:
        roots = [(f"/content/{base_name}", "Colab local (EPHEMERAL)")]
    elif os.path.exists("/kaggle/working"):
        roots = [("/kaggle/working", "Kaggle working")]
    else:
        roots = [("./outputs", "local")]

    root = kind = None
    problems = []
    for cand, label in roots:
        ok, err = _probe_writable(cand)
        if ok:
            root, kind = cand, label
            break
        problems.append(f"{cand}: {err}")

    if root is None:
        raise RuntimeError(
            "No writable output directory found. Tried:\n  " + "\n  ".join(problems) +
            "\n\nRefusing to continue: an earlier version silently fell back to ephemeral "
            "storage here, so runs 'succeeded' and then vanished on disconnect.")

    target = root if kind == "Kaggle working" else os.path.join(root, model_id)
    state = _classify(target)

    if mode == "auto":
        if state == "completed":
            base_dir = _next_version(root, model_id)
            action = f"existing run is COMPLETE -> new version {os.path.basename(base_dir)}"
        elif state == "in_progress":
            base_dir, action = target, "existing run is INCOMPLETE -> reusing it (auto-resume)"
        else:
            base_dir, action = target, f"{state} -> using it"
    elif mode == "new_version":
        if state in ("absent", "empty"):
            base_dir, action = target, f"{state} -> using it"
        else:
            base_dir = _next_version(root, model_id)
            action = f"forced new version -> {os.path.basename(base_dir)}"
    elif mode == "overwrite":
        if state not in ("absent", "empty"):
            shutil.rmtree(target, ignore_errors=True)
            action = "OVERWRITE -> previous contents deleted"
        else:
            action = f"{state} -> using it"
        base_dir = target
    else:  # reuse
        base_dir, action = target, f"{state} -> reusing as requested"

    ok, err = _probe_writable(base_dir)
    if not ok:
        raise RuntimeError(f"Chosen directory {base_dir} is not writable: {err}")

    ckpt_dir = os.path.join(base_dir, "checkpoints")
    results_dir = os.path.join(base_dir, "results")
    for d in (ckpt_dir, results_dir):
        os.makedirs(d, exist_ok=True)

    cache_dir = ("/content/owmtl_spec_cache" if in_colab else
                 "/kaggle/working/owmtl_spec_cache" if os.path.exists("/kaggle/working")
                 else "./owmtl_spec_cache")
    os.makedirs(cache_dir, exist_ok=True)

    info = {"base_dir": base_dir, "ckpt_dir": ckpt_dir, "results_dir": results_dir,
            "cache_dir": cache_dir, "storage": kind, "drive_mounted": mounted,
            "drive_detail": detail, "existing_state": state, "action": action, "mode": mode,
            "is_persistent": "EPHEMERAL" not in kind}

    if verbose:
        print("=" * 70)
        print(f"OUTPUT LOCATION  (mode={mode})")
        print("=" * 70)
        print(f"  storage    : {kind}")
        print(f"  existing   : {state}")
        print(f"  decision   : {action}")
        print(f"  base       : {base_dir}")
        print(f"  checkpoints: {ckpt_dir}")
        print(f"  results    : {results_dir}")
        print(f"  spec cache : {cache_dir}   (local disk, disposable)")
        print(f"  write probe: PASSED")
        if not info["is_persistent"]:
            print("\n  *** WARNING: this is EPHEMERAL storage. Everything is lost when the")
            print("      runtime disconnects. Mount Drive before a long run. ***")
        print("=" * 70)
    return base_dir, info

# ---------------------------------------------------------------- resolve
_OWMTL_ID = (CFG.get("model_id") if isinstance(globals().get("CFG"), dict) else None) or "OWMTL"
BASE_DIR, _OUT_INFO = setup_output_dir(_OWMTL_ID, mode=RUN_MODE)
CKPT_DIR    = _OUT_INFO["ckpt_dir"]
RESULTS_DIR = _OUT_INFO["results_dir"]
CACHE_DIR   = _OUT_INFO["cache_dir"]
IN_COLAB      = "google.colab" in sys.modules or os.path.exists("/content")
DRIVE_MOUNTED = _OUT_INFO["drive_mounted"]

# ---------------------------------------------------------------- rewire CFG
# Every downstream write goes through CFG, so redirecting it here is enough -- no other
# cell needs editing. The spectrogram cache deliberately stays on local disk: it is a
# multi-GB disposable memmap and writing it to Drive would be slow and eat the quota.
if isinstance(globals().get("CFG"), dict):
    _redirect = {"ckpt_dir": CKPT_DIR, "results_dir": RESULTS_DIR,
                 "out_dir": RESULTS_DIR, "cache_dir": CACHE_DIR}
    for _k, _v in _redirect.items():
        if _k in CFG:
            CFG[_k] = _v
            os.makedirs(_v, exist_ok=True)
    print("\nCFG redirected to persistent storage:")
    for _k in ("ckpt_dir", "results_dir", "out_dir", "cache_dir"):
        if _k in CFG:
            print(f"  CFG[{_k!r}] = {CFG[_k]}")

if not _OUT_INFO["is_persistent"]:
    print("\nRefusing to fail silently: outputs are EPHEMERAL. Mount Drive and re-run this")
    print("cell before starting a long training run, or accept that results vanish on")
    print("disconnect. (Change RUN_MODE and re-run this cell only -- nothing else changes.)")


## Section 3 — Model definition and checkpoint

`M3_LightweightCNN` below is byte-identical to the class in `m22-official-notebook.ipynb`. It has
to be: `load_state_dict` matches on parameter names, so any structural edit here silently produces
a differently-wired model that still loads.

Cell 6 **raises** if the checkpoint is missing. It does not fall back to a different checkpoint,
for the same reason the M22_v2 notebook raises on a missing split file — a run that can quietly
produce the wrong answer is worse than one that stops.

In [ ]:
# ============================================================
# CELL 3 — MODEL (verbatim from M22_v2)
# ============================================================
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

class M3_LightweightCNN(nn.Module):
    """1-channel log-mel -> repeat to 3ch -> ImageNet norm -> MobileNetV2 -> GAP -> logits."""
    def __init__(self, arch="mobilenet_v2", num_classes=4, pretrained=True,
                 freeze_features=False, dropout=0.3, imagenet_normalize=True,
                 n_mels=128, n_frames=801):
        super().__init__()
        weights = "IMAGENET1K_V1" if pretrained else None
        try:
            base = getattr(torchvision.models, arch)(weights=weights)
            self.pretrained_loaded = pretrained
        except Exception as e:
            print(f"[warn] pretrained weights unavailable ({e}); random init")
            base = getattr(torchvision.models, arch)(weights=None)
            self.pretrained_loaded = False
        self.arch = arch
        self.imagenet_normalize = bool(imagenet_normalize and self.pretrained_loaded)
        self.needs_relu = arch.startswith("densenet")
        self.features = base.features
        if freeze_features:
            for p in self.features.parameters():
                p.requires_grad = False
        self.features.eval()
        with torch.no_grad():
            self.feature_dim = int(self.features(torch.zeros(1, 3, n_mels, n_frames)).shape[1])
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.feature_dim, num_classes)

    def forward(self, x):
        x = x.repeat(1, 3, 1, 1)
        if self.imagenet_normalize:
            x = (x - IMAGENET_MEAN.to(x.device)) / IMAGENET_STD.to(x.device)
        f = self.features(x)
        if self.needs_relu:
            f = torch.relu(f)
        return self.classifier(self.dropout(self.gap(f).flatten(1)))

model = M3_LightweightCNN(
    arch=CFG["arch"], num_classes=CFG["num_classes"], pretrained=CFG["pretrained"],
    freeze_features=CFG["freeze_features"], dropout=CFG["dropout"],
    imagenet_normalize=CFG["imagenet_normalize"],
    n_mels=CFG["n_mels"], n_frames=CFG["n_frames"]).to(DEVICE)

TOTAL_PARAMS = sum(p.numel() for p in model.parameters())
print(f"{CFG['arch']}: feature_dim={model.feature_dim} | params {TOTAL_PARAMS:,}")

In [ ]:
# ============================================================
# CELL 4 — FIND AND LOAD THE M22_v2 CHECKPOINT
# ============================================================
# Drive holds the checkpoint, so it has to be mounted before this cell globs for it.
# The output-setup cell above already did that on a Run-all. This guard exists because
# people run Colab cells one at a time and out of order, and the failure mode without it
# is "checkpoint candidates found: 0" -- which reads like a missing file rather than an
# unmounted filesystem, and sends you off re-training something you already have.
if PLATFORM == "Colab" and not os.path.isdir("/content/drive/MyDrive"):
    print("Drive is not mounted yet — mounting it now.")
    try:
        from google.colab import drive as _drive
        _drive.mount("/content/drive")
    except Exception as _e:
        print(f"[WARN] could not mount Drive ({_e}). Only local paths will be searched.")

# Where to look, best first. Add a path here if your checkpoint lives somewhere else.
CKPT_HINTS = [
    os.environ.get("M22V2_CKPT", ""),
    "/content/drive/MyDrive/OWMTL/M22_v2/checkpoints/best_model.pth",
    "/content/drive/MyDrive/OWMTL/M22_v2/best_model.pth",
    f"{WORK}/best_model.pth",
]

# Refuses to guess. A wrong checkpoint here silently invalidates every number below,
# and the two wrong ones both look plausible: they load cleanly, they report a
# sensible score, and nothing downstream would notice.
#
# So candidates are not ranked by filename. Each one's stored cfg is read and the
# corrected-split run is preferred; a verbatim-split checkpoint is skipped with a
# note rather than silently used, and the search continues past it.

def _peek(path):
    """(split_method, model_id, epoch, best_score) from a checkpoint's stored cfg."""
    try:
        st = torch.load(path, map_location="cpu", weights_only=False)
    except Exception as e:
        return {"error": str(e)}
    if not isinstance(st, dict):
        return {}
    cfg = st.get("cfg", {}) or {}
    return {"split": str(cfg.get("split_method", "")),
            "model_id": str(cfg.get("model_id", "")),
            "epoch": st.get("epoch"), "best_score": st.get("best_score"), "cfg": cfg}


def candidate_paths():
    seen, out = set(), []
    for h in CKPT_HINTS:
        if h and os.path.exists(h) and h not in seen:
            seen.add(h); out.append(h)
    pats = []
    if PLATFORM == "Kaggle":
        pats += ["/kaggle/input/**/best_model.pth", "/kaggle/input/**/*M22*.pth"]
    if PLATFORM == "Colab":
        pats += ["/content/drive/MyDrive/OWMTL/**/best_model.pth",
                 "/content/drive/MyDrive/**/best_model.pth",
                 "/content/**/best_model.pth"]
    pats += [f"{WORK}/**/best_model.pth"]
    for p in pats:
        for hit in sorted(glob.glob(p, recursive=True)):
            if hit not in seen:
                seen.add(hit); out.append(hit)
    return out


CANDIDATES = candidate_paths()
print(f"checkpoint candidates found: {len(CANDIDATES)}")

CKPT, CKPT_META, _rejected = None, None, []
for c in CANDIDATES:
    meta = _peek(c)
    if meta.get("error"):
        print(f"  skip  {c}\n        unreadable: {meta['error']}")
        continue
    sp = meta.get("split", "")
    if "published_verbatim" in sp:
        _rejected.append((c, "verbatim published split — patients 156 and 218 leak into test"))
        print(f"  skip  {c}\n        {meta.get('model_id')} — leaky published split, not reportable")
        continue
    if sp and "corrected" not in sp:
        _rejected.append((c, f"unrecognised split label {sp!r}"))
        print(f"  skip  {c}\n        unrecognised split label: {sp}")
        continue
    CKPT, CKPT_META = c, meta
    print(f"  USE   {c}")
    break

if CKPT is None:
    _seen = ("\n  Checkpoints that were found and rejected:\n"
             + "".join(f"    {p}\n      -> {why}\n" for p, why in _rejected)) if _rejected else ""
    raise FileNotFoundError(
        "\n" + "=" * 74 +
        "\n  No corrected-split M22_v2 checkpoint found.\n" + "=" * 74 + _seen +
        "\n  This notebook scores the frozen best model. It will not invent one, and it\n"
        "  will not fall back to a checkpoint trained on a partition that leaks.\n"
        "\n  Produce the right one:\n"
        "    1. open  Asif's/M22_v2/m22-official-notebook.ipynb\n"
        "    2. cell 2:  VARIANT = 'augmented'   and   SPLIT_MODE = 'corrected'\n"
        "                'official' is the LEAKY published split. It is not the one.\n"
        "    3. Runtime -> Run all  (~40 epochs, about 2 h on a T4; it resumes if\n"
        "       the runtime drops, because latest.pth is written to Drive every epoch)\n"
        "    4. it saves itself to\n"
        "       /content/drive/MyDrive/OWMTL/M22_v2/checkpoints/best_model.pth\n"
        "       which this cell searches automatically.\n"
        "\n  Do NOT substitute either of these — both load fine and both are wrong:\n"
        "    - Asif's/M22/result_M22/best_model.pth   old fallback split, 11 test patients\n"
        "    - .../M22_v2_official/.../best_model.pth verbatim leaky split\n" + "=" * 74)

state = torch.load(CKPT, map_location=DEVICE, weights_only=False)   # trusted, protocol 11.A
sd = state.get("model_state", state.get("state_dict", state)) if isinstance(state, dict) else state
try:
    model.load_state_dict(sd, strict=True)
except RuntimeError as e:
    raise RuntimeError(
        f"state_dict does not match M3_LightweightCNN:\n{e}\n"
        "The checkpoint was produced by a different architecture. Do not load it "
        "non-strictly — a partially loaded model still runs and still reports a score.") from e
model.eval()

CKPT_EPOCH = int(state.get("epoch", -1))
CKPT_SCORE = float(state.get("best_score", float("nan")))
CKPT_CFG   = state.get("cfg", {}) or {}

print(f"\ncheckpoint : {CKPT}")
print(f"  epoch      {CKPT_EPOCH}")
print(f"  best_score {CKPT_SCORE:.4f}")
print(f"  model_id   {CKPT_CFG.get('model_id', '(not recorded)')}")
print(f"  split      {CKPT_CFG.get('split_method', '(not recorded)')}")
print(f"  specaug    {CKPT_CFG.get('use_specaugment', '(not recorded)')}")

# --- guard rails, re-checked on the loaded file ------------------------------
# _peek already filtered on these, but it read the file separately. Re-asserting here
# means the object actually loaded into the model is the one that passed.
_split = str(CKPT_CFG.get("split_method", ""))
if "published_verbatim" in _split:
    raise RuntimeError(
        f"This checkpoint was trained on the LEAKY published split ({_split}).\n"
        "Part of its test set was seen during training, so a transfer number measured\n"
        "from it is not comparable to the paper's 0.5602. Re-run M22_v2 with\n"
        "SPLIT_MODE = 'corrected'.")
if not CKPT_CFG.get("use_specaugment", True):
    print("[WARN] this checkpoint was trained WITHOUT SpecAugment — that is M3_v2, the")
    print("       control, not the paper's best model. Expect a lower source score.")

# The front-end must match how the checkpoint was trained, or the transfer number
# measures our preprocessing rather than the model.
for k in ("sample_rate", "n_mels", "n_fft", "hop_length", "win_length", "f_min", "f_max"):
    if k in CKPT_CFG and CKPT_CFG[k] != CFG[k]:
        raise RuntimeError(f"preprocessing mismatch on '{k}': "
                           f"checkpoint {CKPT_CFG[k]} vs this notebook {CFG[k]}")
print("preprocessing parameters match the checkpoint.")


## Section 4 — Shared audio front-end

One function, used for both corpora. That is the whole point: if ICBHI and SPRSound went through
different code paths, the comparison would be confounded by the code path.

In [ ]:
# ============================================================
# CELL 5 — LOG-MEL EXTRACTION (verbatim from M22_v2)
# ============================================================
def extract_log_mel(wav_path, start, end, cfg):
    sr, n_samples = cfg["sample_rate"], cfg["n_samples"]
    try:
        audio, _ = librosa.load(wav_path, sr=sr, offset=start,
                                duration=max(end - start, 0.05), mono=True)
    except Exception as e:
        raise RuntimeError(f"failed to load audio: {wav_path}") from e
    if len(audio) == 0:
        raise RuntimeError(f"empty audio decoded from: {wav_path}")

    if len(audio) < n_samples:
        audio = np.tile(audio, math.ceil(n_samples / len(audio)))[:n_samples]
    else:
        audio = audio[:n_samples]

    mel = librosa.feature.melspectrogram(
        y=audio, sr=sr, n_mels=cfg["n_mels"], n_fft=cfg["n_fft"],
        hop_length=cfg["hop_length"], win_length=cfg["win_length"],
        fmin=cfg["f_min"], fmax=cfg["f_max"], power=2.0)
    lm = librosa.power_to_db(mel, ref=np.max)
    lm = (lm - lm.min()) / (lm.max() - lm.min() + 1e-8)

    T = lm.shape[1]
    if T < cfg["n_frames"]:
        lm = np.pad(lm, ((0, 0), (0, cfg["n_frames"] - T)), mode="constant")
    else:
        lm = lm[:, :cfg["n_frames"]]
    return lm[None].astype(np.float32)


def build_cache(frame, tag):
    """float16 memmap so the DataLoader does not re-decode audio every epoch."""
    path = os.path.join(CFG["cache_dir"], f"specs_{tag}_{len(frame)}.npy")
    shape = (len(frame), 1, CFG["n_mels"], CFG["n_frames"])
    if os.path.exists(path):
        print(f"cache hit  : {os.path.basename(path)}")
        return np.memmap(path, dtype=np.float16, mode="r", shape=shape)
    print(f"building cache: {tag} ({len(frame)} segments)")
    mm = np.memmap(path, dtype=np.float16, mode="w+", shape=shape)
    for i, r in enumerate(tqdm(frame.itertuples(), total=len(frame), desc=tag)):
        mm[i] = extract_log_mel(r.wav_path, r.start, r.end, CFG).astype(np.float16)
    mm.flush()
    return np.memmap(path, dtype=np.float16, mode="r", shape=shape)


class SpecDataset(Dataset):
    def __init__(self, specs, labels):
        self.specs, self.labels = specs, labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        return torch.from_numpy(np.asarray(self.specs[i], dtype=np.float32)), int(self.labels[i])


@torch.no_grad()
def infer(specs, labels, desc="inference"):
    """Returns raw logits. Every downstream arm is recomputed from these, so the
    expensive part runs exactly once."""
    ds = SpecDataset(specs, labels)
    dl = DataLoader(ds, batch_size=CFG["batch_size"], shuffle=False,
                    num_workers=CFG["num_workers"], pin_memory=torch.cuda.is_available())
    model.eval()
    out = []
    t0 = time.time()
    for xb, _ in tqdm(dl, desc=desc):
        out.append(model(xb.to(DEVICE, non_blocking=True)).float().cpu())
    logits = torch.cat(out).numpy()
    dt = time.time() - t0
    print(f"{desc}: {len(logits)} segments in {dt:.1f}s "
          f"({1000 * dt / max(len(logits), 1):.2f} ms/segment)")
    return logits, dt

## Section 5 — Metrics

`icbhi_official` is the ICBHI 2017 challenge metric, not the macro variant. The project's audit
measured the macro form running about 0.11 higher; only the official number is comparable to
anything published, and only the official number goes in the paper.

Bootstrap resamples **patients**, not segments. Resampling segments would treat 40 cycles from one
child as 40 independent observations and shrink the interval to a fiction — this is the same error
the paper's own audit section is about.

In [ ]:
# ============================================================
# CELL 6 — METRICS
# ============================================================
def icbhi_official(cm):
    """Se = correct abnormal / all abnormal.  Sp = correct Normal / all Normal.
    Works for the 4-class matrix and, unchanged, for the binary one."""
    cm = np.asarray(cm, dtype=float)
    sp = cm[0, 0] / cm[0].sum() if cm[0].sum() else float("nan")
    abn = cm[1:].sum()
    se = float(np.trace(cm[1:, 1:])) / abn if abn else float("nan")
    return float(se), float(sp), float((se + sp) / 2)


def full_metrics(y_true, y_pred, n_cls, class_names):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(n_cls)))
    p, r, f, sup = precision_recall_fscore_support(
        y_true, y_pred, labels=list(range(n_cls)), zero_division=0)
    spec = []
    for i in range(n_cls):
        tp = cm[i, i]; fp = cm[:, i].sum() - tp; fn = cm[i].sum() - tp
        tn = cm.sum() - tp - fp - fn
        spec.append(float(tn / (tn + fp)) if (tn + fp) else 0.0)
    se_o, sp_o, icbhi_o = icbhi_official(cm)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_macro": float(p.mean()), "recall_macro": float(r.mean()),
        "f1_macro": float(f.mean()), "specificity_macro": float(np.mean(spec)),
        "icbhi_score": float((r.mean() + np.mean(spec)) / 2),   # macro form, never the headline
        "icbhi_score_official": round(icbhi_o, 4),
        "sensitivity_official": round(se_o, 4),
        "specificity_official": round(sp_o, 4),
        "per_class": {class_names[i]: {"precision": round(float(p[i]), 4),
                                       "recall": round(float(r[i]), 4),
                                       "f1": round(float(f[i]), 4),
                                       "specificity": round(spec[i], 4),
                                       "support": int(sup[i])} for i in range(n_cls)},
        "confusion_matrix_raw": cm.astype(int).tolist(),
        "confusion_matrix_normalized": np.round(
            cm / np.clip(cm.sum(1, keepdims=True), 1, None), 4).tolist(),
    }


def patient_bootstrap_ci(y_true, y_pred, groups, n_cls, B=None, seed=SEED):
    """Percentile 95% CI on the official score, resampling PATIENTS with replacement."""
    B = B or CFG["n_bootstrap"]
    y_true, y_pred, groups = map(np.asarray, (y_true, y_pred, groups))
    uniq = np.unique(groups)
    idx_of = {g: np.where(groups == g)[0] for g in uniq}
    rng = np.random.default_rng(seed)
    scores = []
    for _ in range(B):
        pick = rng.choice(uniq, size=len(uniq), replace=True)
        sel = np.concatenate([idx_of[g] for g in pick])
        cm = confusion_matrix(y_true[sel], y_pred[sel], labels=list(range(n_cls)))
        s = icbhi_official(cm)[2]
        if not math.isnan(s):
            scores.append(s)
    if not scores:
        return [float("nan"), float("nan")]
    return [round(float(x), 4) for x in np.percentile(scores, [2.5, 97.5])]


def expected_calibration_error(probs, y_true, n_bins=15):
    probs, y_true = np.asarray(probs), np.asarray(y_true)
    conf, pred = probs.max(1), probs.argmax(1)
    correct = (pred == y_true).astype(float)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum():
            ece += (m.sum() / len(conf)) * abs(correct[m].mean() - conf[m].mean())
    return float(ece)


def softmax(z):
    z = np.asarray(z, dtype=np.float64)
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)

## Section 6 — Sanity gate: does this checkpoint still score 0.5602?

Everything below depends on the checkpoint being the one the paper reports. This cell re-scores it
on the ICBHI corrected test partition and compares against 0.5602.

It needs the ICBHI audio and `ICBHI_challenge_train_test.txt`. On Kaggle attach
`vbookshelf/respiratory-sound-database` plus a dataset holding the split file (the committed copy
is at `Asif's/ICBHI_challenge_train_test.txt`). On Colab point `ICBHI_AUDIO_DIR` and
`ICBHI_SPLIT_FILE` at wherever you mounted them.

Set `REQUIRE_ICBHI_VERIFICATION = False` in Cell 2 to skip. The results JSON then records
`"verified_against_source": false` and the transfer number carries that caveat.

In [ ]:
# ============================================================
# DATA SETUP — credentials, ICBHI audio, official split   [OWMTL_DATA_SETUP_V2]
# ============================================================
# Fully automatic. Nothing to paste, nothing to upload.
#
# ONE-TIME (Colab): left sidebar -> key icon (Secrets) -> add
#     KAGGLE_USERNAME   your kaggle username
#     KAGGLE_KEY        kaggle.com -> Settings -> API -> Create New Token
# Toggle "Notebook access" ON for both. After that every notebook here just works.
#
# The key is never written into the notebook: these files are git-tracked and a Kaggle key is
# full account access. The official split file is EMBEDDED below (3.5 KB gzip+base64), so it
# never needs uploading either.
#
# Sets DATA_ROOT and SPLIT_FILE. Raises with precise instructions if it truly cannot.
import base64 as _b64, glob as _glob, gzip as _gzip, os as _os, subprocess as _sp, sys as _sys

_SPLIT_B64 = "H4sIAAETn2oC/4WcTY/sthFF186vaVUVydLywQiyCAMQsPeDvBcvHuDYDXn+PzKtlnp63vCerA8uWZekqGLpY7ksL8vX5eXL7y9/fXv512//+f66/fnHT6+//fX6t+VgY5sxu+umzF/sq93Yf7+99O+vrz/bL3//6XX79/c/3mA8dXiDbpfLKYynRuesg66DboBuTHVlZ79++2jwMFHfHO6NzhwecG6/vUG/2X+DX/75j59jaf1HuAHspOykHKQcpHwbBAWDrARZCbISZCXISpCVICuFrBSyUshKISuFrBSyUrQVv5q2ssMNYCdlJ+Ug5SClsJLTvemAK+wjK+wjK+wjK+wjK+wjq95Hlst0E72buMGrcnjAjpCaFX3ednUTG9sOXcH5nv8Bdgmvnwf9AxwItyl0vQZOtmnWQddBN0A3XwO3+4zaKU+4AewEBzU7v+yWQjNZ9wUyfgcoFkg74eTOt0NXMGEmE2YyYSYTZjJhJhNmctVpj11guzrhBrATHNTs/Eq3BVKYG7xKaHsK82Wa35xwAzif5R0aNWvUrFGzTs06NavWpLlOjQ2yWIMs1iCLNchiDbJYgyzWKIu1CgYbrdeES9lWUPrtzqSykxNuADspOykHKQcp59nJDp2sOFlxsuJkxcmKkxUnK0FWgqwEWQmyEmQlyEpoK+9bjIIbwE7KTspBykFKsOJkxcmKkxUnK05WnKw4WQmyEmQlyEqQFVhgRheSXQtEey0Q7bVAQNfCfQ5SyrF1mmyngXcaeKexdRpb/woXr9MNwOkG4LSNO23jTtu40zbutEyc9lun/dZpv3XaGP2KYys3xgWyU79nbX2aJLk97X2foL8tzc/zuecQJ9s0G6D70cc7c+jPoT+H/j5P5DPTsQTEEhBLQCwBsYSMxWEeHObBYR5mF9U7U7EEnB5OuElopDRSqjTfCwVUKKAChd4DztPgHRr1adSnUZ9GfTr1iSPk1KeLgvYBRUBVH5edThgHnNe+POkQkXSISDpEJB0ikg4RSYeIpENE0iEi6RCRdIhIOkQkHSKSDhFJh4ikc0LSOSHpnJB0Tkg6JySdE3J63/oIN4CdlJ2Ug5SDlMIKVMIdKuEOlXCHSrhDJdyhEh7PB6wfd4q4wL4Wz+cZAYXy8QhTQbULH7ATxGYhILVFn3ADKPq0c05mEApZEf+X/Vg8urMyPWd/YJtmHdgQzKFNhzZnJ5U7q9Nz+zsTud3JNs066DrohmCpct6Tga4DG9DmPJbnEv+MObCQTK+lBmupwfzNn7N/YJtmHXQddDqWgFgCYgmIJSAWtebpmWqssBms02cUz2zMdOVy32SnbNGHx5NtmnXQddAN0A3QTRfvzhw8OHhw8ODgwcGDgwcHDwEeAjwEeAjwEOAhwENIDw5ryWEtOawXh/XisF4c1ovLeTe4OIsTjKe04scE8YQbwE7KTspBykHKeTpbAmrFJ9wAdlJ2Ug5SDlJKK06z4jQrTrMSNEJBIxQ0QkEjFDRCoUeIii+l6tv0yeatVnjEccJNtzqgx6F6LHoyDyh6LCqlOBn1OEAoQ00Y1aRRTRrVhFFNvRc02rmoDHTATnBQs/MXI0rCOatQdalQdalQdalQdalQdalQdalQdalQdalQdalQdalQdalQdalQdalQdakkPKIuVHoqVF0qVF0qVF0qVF0qVEDaIQyf6UFY6UKi9y7KCvXUssK1Uh/5++er/mSbZh3YgDYH6KZ51c4c4nSI0yFOhzgd4nSIMyDOgDgD4gyIMyDOkHEajItBnAaxGMRiMhZ6dnnA+W2g2nT1foQbwE7KTspBykHK+cVfDd5sqQZvthxwkHKQEgJyCsgpINfDp19BOSEpIdqgaIOiDYo2KNqgaNUNpDpNttNkO7zGdEJSDlIOUkorqZeJQzZ6wE7KTspBSmEl4EZZi36cebBpDexkU12FTwBOuAHspBwI581Stl4pra5UW6xQP9yZfXKyswb5R4P8o0H+0SD/aJB/NMg/GuQfDfKPg3XQddAN0A3QaQ8BHgI8BMQZEGdAnCHjNFgTBmvCYE0YrAmDNWFyTcDHlc2gYnfCDWAnZSflIOUg5XyH32GQlSArQVaCrARZCbISZKWQlUJWClkpZKWQlUJWirZiZMXIilG0RtEaRas/W2zwjLrFtOjyzObbZ0xLLs9sgG6Abn65x7Si8sx0nA5xOsTpEKdDnAFxBsQZEGdAnAFxhoxTPlo72abZAN0A3TyWol+yaUW/ZHOwDroOugG66Us2rUIaUiENqZCGVEhDKqQhFdKQCmlIhTSkQhpSp3eTDwx0A3QDdMqDwTwYzIPBPBjMg8E8GMyDyXlokHo3SL0bpN4NUu8GqTe8d9IarJcG66XBmmiwJhqsCXjPpcF7IA3e52jwPkdruqTWmi6ptYS5TZjbhLlNmNuEuU2Y24T9LGE/S9izEvashD0rYc+6sQpxVoiziuv9YKAboBugUx4M1oTBmjBYEwbzro9O9KjjgPOvWvMCp5G8wGkkL3DgyAscOPICB45czvRg8rQxH99oK2ikVO+9pulXN9OhYH5AMbQBj5gSMqSEDCkhQ0rIkBIypIQMKSuUVk+4AewEBzU7SDk/F2WFInvSGxJZocieFYrsSa9BZIUie1YosmeFIntWKLIfcFCzg5QyWqexdRpbp7F1GlunsXU9tvNPYu5rmiqvmXTVJm0GSZvBCi/IrxdQrgt8VHXCTUL1av1KP61Y6acVK/2XYqX/Uqz064mVfj2xOjyZPOEGsJNyIKRm54tvh+ryfINNXkcnJGUnJfY5SDlIKXyGfkqxhn6Ks8LNb4Wb3wo3vxVufivc/Fa4+a0Vnt+s9OTngOKKzmnZ9N5lTqumz6yDroNugG6AbppxvrF6XaSHel2kh3p/DKcZ6AboBujmHtb7d9zz7WjVX1G9rRD9PdwDbgA7wUHNDlJOL9M7dIrWKVqnaJ2idYrWKdqgaIOiDYo2KNqgaENH6xStU7ROATkF5DogePvnATcJjZRGSielK6X+MapdXL9a+oAbwE5wULODlGLcXb89+oAbwEHKQUoIKCigoICCAgoKKHRARiNkNEJGI2Q0QrDThH7m94AbwK6U7QrNtis0e4PYbCflICgGociS4YNtmnXQddAN0A2hU6WxO9MeDDwYeFBl5BtzUZp+sE0zalP5C1FqfDDQdWAD2pzHUukWUvUXCCcc87uE/qr1wTbNBugG6GZ5oV0aLKcGy0l/1Wrix9HPTMcSEEtALAGxBMQSMhaHOXKYI5fL8M4GtDlAN48z9VuJdoFquC3wU9IHnC7u26+AxVsXJ5t6PJnUObRZJFMf4N6ZAxNzv8AvRW2hTG2hTG2hTG2hTG2hTG2hTG2hTG2hTG2hTG2hTG2Hhfos1GehPgv3KQfBaMqMpsxoVoxmRR9Sl9Angf0HzOrwsRT9E25b9E9bTzamrNEukdBo0iaRcHtcEsJJWUzb2fVzwey90eu88vVgc+GqK8AnFJvBqivAJwSleEZndtGP/u5QlKTNFl3pfsANYCc4dLMYkFGfRn0a9enUp1OfTn2KP+mYwU/uT9gJjjl0vd7N9XVirjcR+EH1yTro+lSn//lzZyb6K+ChQg5ywE5wlmD/D6iHge+6ZgAA"


def _creds():
    """Kaggle credentials, in the order they should already live."""
    try:
        from google.colab import userdata
        u, k = userdata.get("KAGGLE_USERNAME"), userdata.get("KAGGLE_KEY")
        if u and k:
            _os.environ["KAGGLE_USERNAME"], _os.environ["KAGGLE_KEY"] = u.strip(), k.strip()
            return "Colab Secrets"
    except Exception:
        pass
    if _os.environ.get("KAGGLE_USERNAME") and _os.environ.get("KAGGLE_KEY"):
        return "environment variables"
    import json as _j
    p = _os.path.expanduser("~/.kaggle/kaggle.json")
    if _os.path.isfile(p):
        try:
            d = _j.load(open(p))
            if d.get("username") and d.get("key"):
                _os.environ["KAGGLE_USERNAME"] = d["username"]
                _os.environ["KAGGLE_KEY"] = d["key"]
                return "~/.kaggle/kaggle.json"
        except Exception:
            pass
    return None


def _is_audio_dir(d):
    return bool(d) and _os.path.isdir(d) and len(_glob.glob(_os.path.join(d, "*.wav"))) >= 900


# Exact paths first — cheap, and they cover every layout this project has actually seen.
_EXACT = [
    "/content/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files",
    "/content/Respiratory_Sound_Database/audio_and_txt_files",
    "/content/data/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files",
    "/content/audio_and_txt_files",
    "/content/drive/MyDrive/OWMTL_data/audio_and_txt_files",
    "/content/drive/MyDrive/ICBHI/audio_and_txt_files",
    "/kaggle/input/respiratory-sound-database/Respiratory_Sound_Database/"
    "Respiratory_Sound_Database/audio_and_txt_files",
    "./data/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files",
]


def _find_audio():
    for d in _EXACT:
        if _is_audio_dir(d):
            return d
    # Glob only roots that are NOT Drive. A recursive walk of a mounted Drive can take
    # minutes and sometimes hangs on a stale FUSE handle.
    for root in ("/kaggle/input", "/content", "./data", "."):
        if not _os.path.isdir(root):
            continue
        for d in sorted(_glob.glob(_os.path.join(root, "**", "audio_and_txt_files"),
                                   recursive=True)):
            if "/drive/" in d.replace("\\", "/"):
                continue
            if _is_audio_dir(d):
                return d
    return None


def _valid_split(p):
    try:
        rows = [l.split() for l in open(p) if len(l.split()) >= 2]
        return len([r for r in rows if r[1].lower() in ("train", "test")]) == 920
    except Exception:
        return False


def _find_split():
    for root in ("/kaggle/input", "/content", ".", _os.path.expanduser("~")):
        if not _os.path.isdir(root):
            continue
        for pat in ("**/ICBHI_challenge_train_test.txt", "**/*train_test*.txt"):
            for p in sorted(_glob.glob(_os.path.join(root, pat), recursive=True)):
                if "/drive/" in p.replace("\\", "/"):
                    continue
                if _valid_split(p):
                    return p
    return None


print("=" * 70)
print("DATA SETUP")
print("=" * 70)

# ---- 1. ICBHI audio --------------------------------------------------------------
DATA_ROOT = _find_audio()
if DATA_ROOT:
    print(f"ICBHI audio : found -> {DATA_ROOT}")
else:
    _src = _creds()
    print(f"ICBHI audio : not present -> downloading  (credentials: {_src or 'NONE'})")
    if not _src:
        raise RuntimeError(
            "ICBHI audio is missing and no Kaggle credentials were found.\n\n"
            "Set them ONCE: Colab left sidebar -> key icon (Secrets) -> add\n"
            "    KAGGLE_USERNAME   your kaggle username\n"
            "    KAGGLE_KEY        kaggle.com -> Settings -> API -> Create New Token\n"
            "Turn 'Notebook access' ON for BOTH, then re-run this cell.\n"
            "You will never be asked again, in this or any other notebook here.")
    _sp.check_call([_sys.executable, "-m", "pip", "install", "-q", "kaggle"])
    _dest = "/content" if _os.path.isdir("/content") else "./data"
    _os.makedirs(_dest, exist_ok=True)
    try:
        _sp.check_call(["kaggle", "datasets", "download", "-d",
                        "vbookshelf/respiratory-sound-database", "-p", _dest, "--unzip"])
    except _sp.CalledProcessError as e:
        raise RuntimeError(
            f"Kaggle download failed (exit {e.returncode}).\n"
            "Most common cause: the Kaggle account has not accepted the dataset's terms.\n"
            "Open https://www.kaggle.com/datasets/vbookshelf/respiratory-sound-database "
            "once in a browser while signed in, then re-run this cell.") from e
    DATA_ROOT = _find_audio()
    if not DATA_ROOT:
        raise RuntimeError(f"Download finished but no audio_and_txt_files found under {_dest}")
    print(f"ICBHI audio : ready -> {DATA_ROOT}")

_n_wav = len(_glob.glob(_os.path.join(DATA_ROOT, "*.wav")))
_n_txt = len(_glob.glob(_os.path.join(DATA_ROOT, "*.txt")))
print(f"            {_n_wav} wav / {_n_txt} annotation txt")
assert _n_wav >= 900, f"only {_n_wav} wav files under {DATA_ROOT} — download looks incomplete"

# ---- 2. official split -----------------------------------------------------------
# This notebook REFUSES to fall back to a patient-id rule. That silent fallback gave four
# models an 11-patient test set while labelling itself "official 60/40", and correcting it
# is the whole point of the v2 re-runs (Model_Training_Protocol.md section 1).
SPLIT_FILE = _find_split()
if SPLIT_FILE:
    print(f"Split file  : found -> {SPLIT_FILE}")
else:
    for _cand in (_os.path.join(_os.path.dirname(DATA_ROOT.rstrip("/")),
                                "ICBHI_challenge_train_test.txt"),
                  "/content/ICBHI_challenge_train_test.txt",
                  "./ICBHI_challenge_train_test.txt"):
        try:
            with open(_cand, "wb") as _fh:
                _fh.write(_gzip.decompress(_b64.b64decode(_SPLIT_B64)))
            SPLIT_FILE = _cand
            break
        except Exception:
            continue
    if not SPLIT_FILE:
        raise RuntimeError("could not write the embedded split file anywhere")
    print(f"Split file  : written from embedded copy -> {SPLIT_FILE}")

_rows = [l.split() for l in open(SPLIT_FILE) if len(l.split()) >= 2]
_tr = sum(1 for r in _rows if r[1].lower() == "train")
_te = sum(1 for r in _rows if r[1].lower() == "test")
assert (len(_rows), _tr, _te) == (920, 539, 381), \
    f"this is not the official split file: {len(_rows)} recordings, {_tr} train / {_te} test"
print(f"            verified: 920 recordings, {_tr} train / {_te} test")
print("=" * 70)


In [ ]:
# ============================================================
# CELL 7 — ICBHI SANITY GATE
# ============================================================
LABEL_OF = {(0, 0): 0, (1, 0): 1, (0, 1): 2, (1, 1): 3}   # Normal / Crackle / Wheeze / Both
OFFICIAL_OVERLAP = {"156", "218"}    # the two patients on both sides of the published split

def _find_first(patterns):
    for p in patterns:
        hits = sorted(glob.glob(p, recursive=True))
        if hits:
            return hits[0]
    return None

ICBHI_AUDIO_DIR = globals().get("DATA_ROOT") or _find_first([
    "/kaggle/input/**/audio_and_txt_files",
    "/content/drive/MyDrive/**/audio_and_txt_files",
    "/content/**/audio_and_txt_files",
    "./**/audio_and_txt_files",
])
ICBHI_SPLIT_FILE = globals().get("SPLIT_FILE") or _find_first([
    "/kaggle/input/**/ICBHI_challenge_train_test.txt",
    "/content/drive/MyDrive/**/ICBHI_challenge_train_test.txt",
    "/content/**/ICBHI_challenge_train_test.txt",
    "./**/ICBHI_challenge_train_test.txt",
])
print(f"ICBHI audio : {ICBHI_AUDIO_DIR}")
print(f"ICBHI split : {ICBHI_SPLIT_FILE}")

ICBHI_VERIFIED, icbhi_ref = False, None

if ICBHI_AUDIO_DIR is None or ICBHI_SPLIT_FILE is None:
    msg = ("ICBHI audio and/or the official split file were not found, so the checkpoint "
           "cannot be re-verified.")
    if REQUIRE_ICBHI_VERIFICATION:
        raise FileNotFoundError(
            msg + "\nAttach the ICBHI dataset and the split file, or set "
                  "REQUIRE_ICBHI_VERIFICATION = False in Cell 2 and accept an unverified result.")
    print("[WARN] " + msg)
else:
    # --- rebuild the corrected partition, protocol section 1 --------------------
    raw_split = {}
    with open(ICBHI_SPLIT_FILE) as fh:
        for line in fh:
            parts = line.split()
            if len(parts) >= 2:
                raw_split[parts[0]] = parts[1].strip().lower()
    assert len(raw_split) == 920, f"split file has {len(raw_split)} recordings, expected 920"

    def patient_id_from_stem(stem):
        return stem.split("_")[0]

    by_patient = {}
    for stem, side in raw_split.items():
        by_patient.setdefault(patient_id_from_stem(stem), set()).add(side)
    leaking = {p for p, sides in by_patient.items() if len(sides) > 1}
    assert leaking == OFFICIAL_OVERLAP, f"expected leaking patients {OFFICIAL_OVERLAP}, got {leaking}"
    print(f"leaking patients in the published split: {sorted(leaking)} -> reassigned to train")

    corrected = {s: ("train" if patient_id_from_stem(s) in leaking else side)
                 for s, side in raw_split.items()}

    def parse_annotation(txt_path):
        out = []
        with open(txt_path) as fh:
            for line in fh:
                p = line.split()
                if len(p) >= 4:
                    out.append({"start": float(p[0]), "end": float(p[1]),
                                "label": LABEL_OF[(int(p[2]), int(p[3]))]})
        return out

    rows = []
    for wav in sorted(glob.glob(os.path.join(ICBHI_AUDIO_DIR, "*.wav"))):
        stem = os.path.splitext(os.path.basename(wav))[0]
        txt = os.path.join(ICBHI_AUDIO_DIR, stem + ".txt")
        side = corrected.get(stem)
        if not os.path.exists(txt) or side != "test":
            continue
        for c in parse_annotation(txt):
            rows.append({"wav_path": wav, "stem": stem,
                         "patient_id": patient_id_from_stem(stem),
                         "start": c["start"], "end": c["end"], "label": c["label"]})
    df_icbhi = pd.DataFrame(rows)
    n_cyc, n_pat = len(df_icbhi), df_icbhi.patient_id.nunique()
    print(f"ICBHI corrected test: {n_cyc} cycles / {n_pat} patients "
          f"(paper: {CFG['source_test_cycles']} / {CFG['source_test_patients']})")
    assert n_cyc == CFG["source_test_cycles"], \
        f"rebuilt {n_cyc} test cycles, paper reports {CFG['source_test_cycles']}"

    specs_icbhi = build_cache(df_icbhi, "icbhi_test")
    y_icbhi = df_icbhi.label.values.astype(np.int64)
    logits_icbhi, _ = infer(specs_icbhi, y_icbhi, "ICBHI test")
    pred_icbhi = logits_icbhi.argmax(1)
    probs_icbhi = softmax(logits_icbhi)

    icbhi_ref = full_metrics(y_icbhi, pred_icbhi, 4, CFG["classes"])
    icbhi_ref["ci95"] = patient_bootstrap_ci(y_icbhi, pred_icbhi,
                                             df_icbhi.patient_id.values, 4)
    icbhi_ref["ece"] = round(expected_calibration_error(probs_icbhi, y_icbhi), 4)
    icbhi_ref["mean_max_softmax"] = round(float(probs_icbhi.max(1).mean()), 4)
    icbhi_ref["n_cycles"] = int(n_cyc)
    icbhi_ref["n_patients"] = int(n_pat)
    icbhi_ref["class_prior"] = (np.bincount(y_icbhi, minlength=4) / len(y_icbhi)).round(4).tolist()

    got, want = icbhi_ref["icbhi_score_official"], CFG["source_score_icbhi"]
    delta = got - want
    print("\n" + "=" * 62)
    print(f"  SANITY GATE   reproduced {got:.4f}   paper {want:.4f}   delta {delta:+.4f}")
    print("=" * 62)
    if abs(delta) <= 0.0005:
        ICBHI_VERIFIED = True
        print("  PASS — this is the checkpoint the paper reports.")
    elif abs(delta) <= 0.01:
        ICBHI_VERIFIED = True
        print("  PASS (within tolerance). Small drift is possible across library versions;")
        print("  the exact delta is recorded in the results JSON.")
    else:
        raise RuntimeError(
            f"SANITY GATE FAILED: {got:.4f} vs {want:.4f} (delta {delta:+.4f}).\n"
            "This is not the checkpoint behind the paper's 0.5602, or the front-end differs.\n"
            "Do not report a transfer number from it.")

## Section 7 — SPRSound: acquisition and parsing

SPRSound ships one JSON per recording holding an `event_annotation` list (start, end, type) and a
record-level annotation. The parser below does **not** assume the exact key names or the time unit.
It discovers them, prints what it found, and raises on anything it cannot map — an unmapped event
type silently dropped would quietly change the denominator of the transfer score.

Times are detected as milliseconds or seconds by comparing against the actual audio duration,
rather than trusted from the schema.

In [ ]:
# ============================================================
# CELL 8 — ACQUIRE SPRSOUND
# ============================================================
SPR_ROOT = SPRSOUND_LOCAL

if SPR_ROOT is None:
    for cand in ["/kaggle/input", "/content/drive/MyDrive", WORK, "."]:
        hits = sorted(glob.glob(os.path.join(cand, "**", "SPRSound*"), recursive=True))
        hits = [h for h in hits if os.path.isdir(h)]
        if hits:
            SPR_ROOT = hits[0]
            print(f"found an existing SPRSound copy: {SPR_ROOT}")
            break

if SPR_ROOT is None:
    SPR_ROOT = os.path.join(WORK, "SPRSound")
    print(f"cloning SPRSound into {SPR_ROOT} (a few minutes) ...")
    r = subprocess.run(["git", "clone", "--depth", "1", SPRSOUND_GIT, SPR_ROOT],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(
            f"git clone failed:\n{r.stderr}\n\n"
            "Download SPRSound another way (GitHub zip, or attach it as a Kaggle dataset) "
            "and set SPRSOUND_LOCAL in Cell 2 to that folder.")

wavs  = sorted(glob.glob(os.path.join(SPR_ROOT, "**", "*.wav"),  recursive=True))
jsons = sorted(glob.glob(os.path.join(SPR_ROOT, "**", "*.json"), recursive=True))
print(f"SPRSound root : {SPR_ROOT}")
print(f"  {len(wavs)} wav files, {len(jsons)} json files")
if not wavs or not jsons:
    raise RuntimeError(f"no wav/json pairs under {SPR_ROOT} — check the download")

print("\nfirst few wav stems (check the patient-id convention below matches):")
for w in wavs[:5]:
    print("   ", os.path.basename(w))
print("\nschema of the first annotation file:")
with open(jsons[0]) as fh:
    _probe = json.load(fh)
print("   ", json.dumps(_probe, indent=1)[:600])

In [ ]:
# ============================================================
# CELL 9 — PARSE SPRSOUND EVENTS
# ============================================================
# Key names, time units and label spellings are all DISCOVERED, not assumed.

EVENT_KEYS  = ("event_annotation", "events", "event", "annotation")
RECORD_KEYS = ("record_annotation", "record", "record_label")
START_KEYS  = ("start", "start_time", "begin")
END_KEYS    = ("end", "end_time", "stop")
TYPE_KEYS   = ("type", "label", "event_type", "class")

def _pick(d, keys):
    for k in keys:
        if k in d:
            return d[k]
    lower = {str(k).lower(): v for k, v in d.items()}
    for k in keys:
        if k in lower:
            return lower[k]
    return None

json_by_stem = {os.path.splitext(os.path.basename(j))[0]: j for j in jsons}
wav_by_stem  = {os.path.splitext(os.path.basename(w))[0]: w for w in wavs}
paired = sorted(set(json_by_stem) & set(wav_by_stem))
print(f"paired wav+json recordings: {len(paired)} "
      f"(wav only {len(set(wav_by_stem) - set(json_by_stem))}, "
      f"json only {len(set(json_by_stem) - set(wav_by_stem))})")
if not paired:
    raise RuntimeError("no wav/json stems matched — inspect the printout in Cell 8")

rows, record_labels, n_no_events, unit_votes = [], {}, 0, {"s": 0, "ms": 0}

for stem in tqdm(paired, desc="parsing annotations"):
    with open(json_by_stem[stem]) as fh:
        ann = json.load(fh)
    if not isinstance(ann, dict):
        continue
    rec = _pick(ann, RECORD_KEYS)
    record_labels[stem] = str(rec) if rec is not None else "(none)"

    events = _pick(ann, EVENT_KEYS)
    if not isinstance(events, list) or not events:
        n_no_events += 1
        continue

    wav = wav_by_stem[stem]
    try:
        dur = float(librosa.get_duration(path=wav))
    except TypeError:                       # librosa < 0.10 named it filename=
        dur = float(librosa.get_duration(filename=wav))
    except Exception:
        dur = float("nan")

    parsed = []
    for ev in events:
        if not isinstance(ev, dict):
            continue
        s, e, t = _pick(ev, START_KEYS), _pick(ev, END_KEYS), _pick(ev, TYPE_KEYS)
        if s is None or e is None or t is None:
            continue
        try:
            parsed.append((float(s), float(e), str(t).strip()))
        except (TypeError, ValueError):
            continue
    if not parsed:
        n_no_events += 1
        continue

    # time unit: milliseconds or seconds, decided against the real duration
    max_end = max(p[1] for p in parsed)
    if math.isnan(dur) or dur <= 0:
        scale = 1000.0 if max_end > 1000 else 1.0
    elif max_end > dur * 1.5:
        scale = 1000.0
    else:
        scale = 1.0
    unit_votes["ms" if scale == 1000.0 else "s"] += 1

    for s, e, t in parsed:
        rows.append({"wav_path": wav, "stem": stem,
                     "start": s / scale, "end": e / scale,
                     "event_type": t, "record_label": record_labels[stem]})

spr = pd.DataFrame(rows)
print(f"\nrecordings with no usable event list : {n_no_events}")
print(f"time units detected                  : {unit_votes}")
print(f"raw events parsed                    : {len(spr)}")
if unit_votes["s"] and unit_votes["ms"]:
    print("[WARN] mixed time units across files — each file was scaled on its own duration")
if spr.empty:
    raise RuntimeError("no events parsed — inspect the schema printed in Cell 8")

print("\nobserved event types:")
for t, n in spr.event_type.value_counts().items():
    print(f"   {t:<24} {n:6d}")
print("\nobserved record-level labels:")
for t, n in pd.Series(record_labels).value_counts().items():
    print(f"   {t:<24} {n:6d}")

In [ ]:
# ============================================================
# CELL 10 — LABEL MAPPING (two mappings, both reported)
# ============================================================
# ICBHI classes:  0 Normal | 1 Crackle | 2 Wheeze | 3 Both
#
# STRICT  only the event types with an unambiguous ICBHI counterpart.
#         Rhonchi and Stridor are DROPPED. Rhonchi is a low-pitched continuous sound and
#         stridor is an upper-airway inspiratory one; ICBHI's "wheeze" class was not
#         annotated to cover either, so folding them in would score the model against a
#         label it was never taught.
# BROAD   the clinical superset: every continuous adventitious sound counts as Wheeze.
#         This is how several SPRSound papers collapse the taxonomy.
#
# Both are reported. If they disagree materially, the disagreement is itself the finding.

def _norm(s):
    return "".join(ch for ch in str(s).lower() if ch.isalnum())

STRICT_MAP = {
    "normal":            0,
    "finecrackle":       1,
    "coarsecrackle":     1,
    "crackle":           1,
    "wheeze":            2,
    "wheezecrackle":     3,
    "wheezeandcrackle":  3,
    "crackleandwheeze":  3,
}
BROAD_MAP = dict(STRICT_MAP)
BROAD_MAP.update({"rhonchi": 2, "stridor": 2})

# Anything the corpus contains that neither map covers must be seen, not silently dropped.
observed = {_norm(t) for t in spr.event_type.unique()}
unmapped = sorted(observed - set(BROAD_MAP))
if unmapped:
    print("[!] event types not covered by either mapping:")
    for u in unmapped:
        n = int((spr.event_type.map(_norm) == u).sum())
        print(f"      {u!r}  ({n} events)")
    print("    They are excluded from BOTH arms and counted in the results JSON.")

spr["_norm"] = spr.event_type.map(_norm)
spr["label_strict"] = spr._norm.map(STRICT_MAP)
spr["label_broad"]  = spr._norm.map(BROAD_MAP)

# --- quality filter ---------------------------------------------------------
# SPRSound marks unusable recordings at the record level. Scoring them would measure
# the corpus's own noise floor, not transfer.
_poor = spr.record_label.map(lambda s: "poor" in str(s).lower())
n_poor = int(_poor.sum())
spr = spr[~_poor].copy()
print(f"dropped {n_poor} events from recordings marked poor quality")

# --- patient id and age from the filename -----------------------------------
# SPRSound stems are underscore-delimited and lead with the patient number. Parsed
# defensively and printed so it can be eyeballed against Cell 8's stem list.
def spr_patient_id(stem):
    return str(stem).split("_")[0]

def spr_age(stem):
    parts = str(stem).split("_")
    for p in parts[1:4]:
        try:
            a = float(p)
            if 0.0 <= a <= 20.0:
                return a
        except ValueError:
            continue
    return float("nan")

spr["patient_id"] = spr.stem.map(spr_patient_id)
spr["age"] = spr.stem.map(spr_age)

spr = spr[(spr.end > spr.start) & (spr.end - spr.start >= 0.05)].reset_index(drop=True)

print(f"\nusable events   : {len(spr)}")
print(f"recordings      : {spr.stem.nunique()}")
print(f"patients        : {spr.patient_id.nunique()}")
print(f"age parsed for  : {int(spr.age.notna().sum())} / {len(spr)} events "
      f"(range {spr.age.min():.1f}-{spr.age.max():.1f})" if spr.age.notna().any()
      else "age could not be parsed from the filenames")
print(f"segment length  : median {(spr.end - spr.start).median():.2f}s, "
      f"max {(spr.end - spr.start).max():.2f}s")

for name, col in (("STRICT", "label_strict"), ("BROAD", "label_broad")):
    sub = spr[spr[col].notna()]
    print(f"\n{name} mapping — {len(sub)} events, {sub.patient_id.nunique()} patients")
    for i, cname in enumerate(CFG["classes"]):
        n = int((sub[col] == i).sum())
        print(f"   {cname:<9} {n:6d}  ({100 * n / max(len(sub), 1):5.1f}%)")

## Section 8 — Inference on SPRSound

One forward pass over the union of both mappings. The two arms are then scored by subsetting the
stored logits, so nothing is inferred twice.

In [ ]:
# ============================================================
# CELL 11 — SPRSOUND INFERENCE
# ============================================================
spr_eval = spr[spr.label_broad.notna()].reset_index(drop=True)   # superset of STRICT
print(f"scoring {len(spr_eval)} events from {spr_eval.patient_id.nunique()} patients")

specs_spr = build_cache(spr_eval, "sprsound")
logits_spr, spr_infer_s = infer(specs_spr,
                                spr_eval.label_broad.values.astype(np.int64), "SPRSound")
probs_spr = softmax(logits_spr)
pred_spr  = logits_spr.argmax(1)

np.save(os.path.join(CFG["out_dir"], "logits_M49_sprsound.npy"), logits_spr)
spr_eval.drop(columns=["_norm"]).to_csv(
    os.path.join(CFG["out_dir"], "index_M49_sprsound.csv"), index=False)
print("logits and index written — every arm below is recomputable without re-running inference")

## Section 9 — Arm 1: zero-shot 4-class transfer

The headline. Frozen weights, unseen corpus, unseen population, unseen device.

In [ ]:
# ============================================================
# CELL 12 — ARM 1: ZERO-SHOT 4-CLASS
# ============================================================
ARMS = {}

def score_arm(name, mask, y, pred, groups, n_cls, class_names, note=""):
    m = np.asarray(mask, dtype=bool)
    y, pred, groups = np.asarray(y)[m], np.asarray(pred)[m], np.asarray(groups)[m]
    res = full_metrics(y, pred, n_cls, class_names)
    res["ci95"] = patient_bootstrap_ci(y, pred, groups, n_cls)
    res["n_events"] = int(m.sum())
    res["n_patients"] = int(len(np.unique(groups)))
    res["note"] = note
    ARMS[name] = res
    print(f"\n--- {name} ---")
    if note:
        print(f"    {note}")
    print(f"    {res['n_events']} events / {res['n_patients']} patients")
    print(f"    ICBHI score {res['icbhi_score_official']:.4f}  CI {res['ci95']}"
          f"   Se {res['sensitivity_official']:.4f}  Sp {res['specificity_official']:.4f}")
    print(f"    accuracy {res['accuracy']:.4f}   macro-F1 {res['f1_macro']:.4f}")
    return res

groups_spr = spr_eval.patient_id.values

_strict = spr_eval.label_strict.notna().values
score_arm("zeroshot_4class_strict", _strict,
          spr_eval.label_strict.fillna(0).values.astype(np.int64), pred_spr, groups_spr,
          4, CFG["classes"],
          "unambiguous event types only; rhonchi and stridor excluded")

score_arm("zeroshot_4class_broad", np.ones(len(spr_eval), bool),
          spr_eval.label_broad.values.astype(np.int64), pred_spr, groups_spr,
          4, CFG["classes"],
          "rhonchi and stridor folded into Wheeze as continuous adventitious sounds")

## Section 10 — Arm 2: binary transfer

Four-class agreement across corpora asks a lot of two annotation protocols that were written
independently. Normal against adventitious is the distinction both corpora genuinely share, so it
is the fairer test of whether the representation transferred at all. The paper already reports the
binary collapse on ICBHI, so the two are directly comparable.

In [ ]:
# ============================================================
# CELL 13 — ARM 2: BINARY (Normal vs adventitious)
# ============================================================
BIN_NAMES = ["Normal", "Adventitious"]

y_bin_strict = (spr_eval.label_strict.fillna(-1).values > 0).astype(np.int64)
y_bin_broad  = (spr_eval.label_broad.values > 0).astype(np.int64)
pred_bin     = (pred_spr > 0).astype(np.int64)

score_arm("zeroshot_binary_strict", _strict, y_bin_strict, pred_bin, groups_spr,
          2, BIN_NAMES, "Normal vs any adventitious sound, strict event set")
score_arm("zeroshot_binary_broad", np.ones(len(spr_eval), bool), y_bin_broad, pred_bin,
          groups_spr, 2, BIN_NAMES, "Normal vs any adventitious sound, broad event set")

if ICBHI_VERIFIED:
    y_icbhi_bin = (y_icbhi > 0).astype(np.int64)
    p_icbhi_bin = (pred_icbhi > 0).astype(np.int64)
    icbhi_bin = full_metrics(y_icbhi_bin, p_icbhi_bin, 2, BIN_NAMES)
    icbhi_bin["ci95"] = patient_bootstrap_ci(y_icbhi_bin, p_icbhi_bin,
                                             df_icbhi.patient_id.values, 2)
    print(f"\n--- reference: same model, binary, on ICBHI ---")
    print(f"    ICBHI score {icbhi_bin['icbhi_score_official']:.4f}  CI {icbhi_bin['ci95']}")
else:
    icbhi_bin = None

## Section 11 — Arm 3: reference baselines on SPRSound

A transfer score cannot be read on its own. On ICBHI, always predicting Normal scores 0.5918
*accuracy* — higher than our best model — which is exactly why this project reports the official
metric and the trivial baselines side by side. The same discipline applies here.

Note that always-Normal has an official score of exactly 0.5 by construction (Se 0, Sp 1). It is
listed because it is the floor a reviewer will reach for, not because it is informative.

In [ ]:
# ============================================================
# CELL 14 — ARM 3: TRIVIAL BASELINES ON SPRSOUND
# ============================================================
rng = np.random.default_rng(SEED)
y4 = spr_eval.label_broad.values.astype(np.int64)
prior_spr = np.bincount(y4, minlength=4) / len(y4)

BASELINES = {}
def _bl(name, pred):
    cm = confusion_matrix(y4, pred, labels=[0, 1, 2, 3])
    se, sp, sc = icbhi_official(cm)
    BASELINES[name] = {"icbhi_score_official": round(sc, 4),
                       "sensitivity_official": round(se, 4),
                       "specificity_official": round(sp, 4),
                       "accuracy": round(float(accuracy_score(y4, pred)), 4)}
    print(f"  {name:<28} score {sc:.4f}  Se {se:.4f}  Sp {sp:.4f}  acc {accuracy_score(y4, pred):.4f}")

print("reference baselines on the SPRSound broad arm:")
_bl("always Normal",            np.zeros(len(y4), np.int64))
_bl("always majority class",    np.full(len(y4), int(np.argmax(prior_spr)), np.int64))
_bl("uniform random",           rng.integers(0, 4, len(y4)))
_bl("prior-matched random",     rng.choice(4, size=len(y4), p=prior_spr))
_bl("our model (broad)",        pred_spr)

print(f"\nSPRSound class prior: "
      + ", ".join(f"{c} {p:.3f}" for c, p in zip(CFG['classes'], prior_spr)))
if ICBHI_VERIFIED:
    print("ICBHI test prior    : "
          + ", ".join(f"{c} {p:.3f}" for c, p in zip(CFG['classes'], icbhi_ref['class_prior'])))

## Section 12 — Arm 4: how much of the drop is prior shift?

A classifier moved to a corpus with a different class balance loses score even when its features
transferred perfectly. Subtracting the source log-prior and adding the target log-prior removes
that component and leaves the part attributable to the representation.

This uses the target prior, so it is **not** a zero-shot result — it is a diagnostic upper bound,
and it is labelled as one everywhere it appears. Report it as decomposition, never as the headline.

In [ ]:
# ============================================================
# CELL 15 — ARM 4: PRIOR-SHIFT CORRECTION (diagnostic, not zero-shot)
# ============================================================
if ICBHI_VERIFIED:
    prior_src = np.asarray(icbhi_ref["class_prior"], dtype=np.float64)
    src_note = "source prior measured on the ICBHI corrected test partition"
else:
    prior_src = np.array([0.5918, 0.2667, 0.0326, 0.1089])   # published ICBHI test balance
    src_note = "source prior taken from the published ICBHI test balance (ICBHI not attached)"
prior_src = np.clip(prior_src, 1e-6, None); prior_src /= prior_src.sum()
prior_tgt = np.clip(prior_spr, 1e-6, None); prior_tgt /= prior_tgt.sum()

adj = logits_spr - np.log(prior_src)[None, :] + np.log(prior_tgt)[None, :]
pred_adj = adj.argmax(1)

print(src_note)
score_arm("prior_corrected_4class_broad", np.ones(len(spr_eval), bool),
          y4, pred_adj, groups_spr, 4, CFG["classes"],
          "DIAGNOSTIC ONLY — uses the target class prior, so it is not a zero-shot number")

_d = (ARMS["prior_corrected_4class_broad"]["icbhi_score_official"]
      - ARMS["zeroshot_4class_broad"]["icbhi_score_official"])
print(f"\nprior correction moves the broad arm by {_d:+.4f}")
print("Read: the part of the transfer gap that is class-balance shift rather than")
print("      a failure of the learned representation.")

## Section 13 — Arm 5: confidence and calibration under shift

For a paper about trustworthy evaluation this matters as much as the score. A model that fails
quietly, keeping high confidence on a corpus it cannot classify, is worse than one that fails
loudly. This measures which of the two we have.

In [ ]:
# ============================================================
# CELL 16 — ARM 5: CALIBRATION
# ============================================================
def conf_report(probs, y, tag):
    conf = probs.max(1)
    ent  = -(probs * np.log(np.clip(probs, 1e-12, None))).sum(1)
    ece  = expected_calibration_error(probs, y)
    acc  = float((probs.argmax(1) == np.asarray(y)).mean())
    d = {"ece": round(ece, 4), "mean_max_softmax": round(float(conf.mean()), 4),
         "median_max_softmax": round(float(np.median(conf)), 4),
         "mean_entropy": round(float(ent.mean()), 4),
         "accuracy": round(acc, 4),
         "overconfidence": round(float(conf.mean() - acc), 4)}
    print(f"  {tag:<12} ECE {d['ece']:.4f} | mean conf {d['mean_max_softmax']:.4f} "
          f"| acc {d['accuracy']:.4f} | overconfidence {d['overconfidence']:+.4f} "
          f"| entropy {d['mean_entropy']:.4f}")
    return d

print("calibration:")
CAL = {"sprsound": conf_report(probs_spr, y4, "SPRSound")}
if ICBHI_VERIFIED:
    CAL["icbhi"] = conf_report(probs_icbhi, y_icbhi, "ICBHI")
    CAL["delta_ece"] = round(CAL["sprsound"]["ece"] - CAL["icbhi"]["ece"], 4)
    CAL["delta_overconfidence"] = round(
        CAL["sprsound"]["overconfidence"] - CAL["icbhi"]["overconfidence"], 4)
    print(f"\n  ECE change on transfer            {CAL['delta_ece']:+.4f}")
    print(f"  overconfidence change on transfer {CAL['delta_overconfidence']:+.4f}")
    print("\n  A large positive overconfidence change means the model keeps its confidence")
    print("  while losing accuracy — it fails silently on the new corpus.")

## Section 14 — Arm 6: does the drop track age?

Gap7 found that the physics-informed loss (M35) hurt paediatric generalization, and read that as
adult-tuned acoustic priors failing on children's higher resonant frequencies. SPRSound carries
age, so that explanation is testable here rather than asserted: if the drop is driven by paediatric
airway acoustics, score should rise with age.

In [ ]:
# ============================================================
# CELL 17 — ARM 6: AGE STRATIFICATION
# ============================================================
AGE_STRATA = None
if spr_eval.age.notna().any():
    bands = [(0, 1, "under 1"), (1, 3, "1-3"), (3, 6, "3-6"),
             (6, 12, "6-12"), (12, 100, "12+")]
    AGE_STRATA = []
    print(f"{'band':<10} {'n':>6} {'patients':>9} {'score':>8} {'Se':>7} {'Sp':>7}")
    print("-" * 52)
    for lo, hi, name in bands:
        m = ((spr_eval.age >= lo) & (spr_eval.age < hi)).to_numpy(dtype=bool)
        if m.sum() < 30:
            continue
        cm = confusion_matrix(y4[m], pred_spr[m], labels=[0, 1, 2, 3])
        se, sp, sc = icbhi_official(cm)
        npat = int(len(np.unique(spr_eval.patient_id.values[m])))
        AGE_STRATA.append({"band": name, "n_events": int(m.sum()), "n_patients": npat,
                           "icbhi_score_official": round(sc, 4),
                           "sensitivity_official": round(se, 4),
                           "specificity_official": round(sp, 4)})
        print(f"{name:<10} {m.sum():6d} {npat:9d} {sc:8.4f} {se:7.4f} {sp:7.4f}")
    if len(AGE_STRATA) >= 3:
        xs = np.arange(len(AGE_STRATA), dtype=float)
        ys = np.array([a["icbhi_score_official"] for a in AGE_STRATA])
        slope = float(np.polyfit(xs, ys, 1)[0])
        print(f"\ntrend across bands: {slope:+.4f} score per band")
        print("A positive slope supports the paediatric-acoustics reading of Gap7;")
        print("a flat one says the gap is corpus or device, not age.")
else:
    print("age was not parsable from the SPRSound filenames — stratification skipped")

## Section 15 — Figures

In [ ]:
# ============================================================
# CELL 18 — FIGURES
# ============================================================
matplotlib.rcParams.update({"font.size": 9, "pdf.fonttype": 42, "figure.dpi": 130})

def plot_cm(ax, cm, names, title):
    cm = np.asarray(cm, float)
    norm = cm / np.clip(cm.sum(1, keepdims=True), 1, None)
    im = ax.imshow(norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=45, ha="right")
    ax.set_yticks(range(len(names))); ax.set_yticklabels(names)
    for i in range(len(names)):
        for j in range(len(names)):
            ax.text(j, i, f"{norm[i, j]:.2f}\n({int(cm[i, j])})", ha="center", va="center",
                    fontsize=7, color="white" if norm[i, j] > 0.5 else "black")
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(title, fontsize=9)
    return im

# --- confusion matrices -----------------------------------------------------
n_panels = 3 if ICBHI_VERIFIED else 2
fig, axes = plt.subplots(1, n_panels, figsize=(4.2 * n_panels, 3.8))
k = 0
if ICBHI_VERIFIED:
    plot_cm(axes[k], icbhi_ref["confusion_matrix_raw"], CFG["classes"],
            f"ICBHI test\nscore {icbhi_ref['icbhi_score_official']:.4f}"); k += 1
plot_cm(axes[k], ARMS["zeroshot_4class_broad"]["confusion_matrix_raw"], CFG["classes"],
        f"SPRSound, 4-class broad\nscore {ARMS['zeroshot_4class_broad']['icbhi_score_official']:.4f}"); k += 1
plot_cm(axes[k], ARMS["zeroshot_binary_broad"]["confusion_matrix_raw"], BIN_NAMES,
        f"SPRSound, binary\nscore {ARMS['zeroshot_binary_broad']['icbhi_score_official']:.4f}")
fig.suptitle("MobileNetV2 + SpecAugment, frozen — same weights on both corpora", fontsize=10)
fig.tight_layout()
fig.savefig(os.path.join(CFG["out_dir"], "M49_confusion_matrices.png"),
            dpi=300, bbox_inches="tight")
plt.show()

# --- score comparison with CIs ---------------------------------------------
labels, vals, los, his = [], [], [], []
if ICBHI_VERIFIED:
    labels.append("ICBHI\n(source)"); vals.append(icbhi_ref["icbhi_score_official"])
    los.append(icbhi_ref["ci95"][0]); his.append(icbhi_ref["ci95"][1])
for key, lab in [("zeroshot_4class_strict", "SPRSound\n4-class strict"),
                 ("zeroshot_4class_broad",  "SPRSound\n4-class broad"),
                 ("zeroshot_binary_broad",  "SPRSound\nbinary"),
                 ("prior_corrected_4class_broad", "SPRSound\nprior-corrected*")]:
    labels.append(lab); vals.append(ARMS[key]["icbhi_score_official"])
    los.append(ARMS[key]["ci95"][0]); his.append(ARMS[key]["ci95"][1])

fig, ax = plt.subplots(figsize=(1.35 * len(labels) + 1.5, 3.8))
x = np.arange(len(labels))
err = np.vstack([np.array(vals) - np.array(los), np.array(his) - np.array(vals)])
cols = ["#2a78d6" if "ICBHI" in l else "#eb6834" for l in labels]
ax.bar(x, vals, color=cols, width=0.62)
ax.errorbar(x, vals, yerr=np.clip(err, 0, None), fmt="none", ecolor="#0b0b0b",
            capsize=4, lw=1.1)
ax.axhline(0.5, color="#8a8985", ls="--", lw=1)
ax.text(len(labels) - 0.45, 0.507, "chance", fontsize=7.5, color="#52514e", ha="right")
for xi, v in zip(x, vals):
    ax.text(xi, v + 0.012, f"{v:.4f}", ha="center", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=8)
ax.set_ylabel("ICBHI challenge score")
ax.set_ylim(0, max(his + [0.7]) * 1.18)
ax.set_title("Zero-shot transfer, frozen weights (bars: 95% patient bootstrap)", fontsize=9.5)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.03, "* diagnostic only — uses the target class prior, not a zero-shot result",
         fontsize=7, color="#52514e")
fig.tight_layout()
fig.savefig(os.path.join(CFG["out_dir"], "M49_transfer_scores.png"),
            dpi=300, bbox_inches="tight")
plt.show()

# --- confidence histograms --------------------------------------------------
fig, ax = plt.subplots(figsize=(5.2, 3.4))
bins = np.linspace(0.25, 1.0, 31)
if ICBHI_VERIFIED:
    ax.hist(probs_icbhi.max(1), bins=bins, alpha=0.62, label="ICBHI (source)",
            color="#2a78d6", density=True)
ax.hist(probs_spr.max(1), bins=bins, alpha=0.62, label="SPRSound (transfer)",
        color="#eb6834", density=True)
ax.set_xlabel("max softmax probability"); ax.set_ylabel("density")
ax.set_title("Confidence under domain shift", fontsize=9.5)
ax.legend(frameon=False, fontsize=8)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(os.path.join(CFG["out_dir"], "M49_confidence.png"), dpi=300, bbox_inches="tight")
plt.show()

# --- age strata -------------------------------------------------------------
if AGE_STRATA and len(AGE_STRATA) >= 2:
    fig, ax = plt.subplots(figsize=(5.2, 3.2))
    xs = np.arange(len(AGE_STRATA))
    ax.plot(xs, [a["icbhi_score_official"] for a in AGE_STRATA], "o-",
            color="#eb6834", lw=1.6)
    ax.axhline(0.5, color="#8a8985", ls="--", lw=1)
    if ICBHI_VERIFIED:
        ax.axhline(icbhi_ref["icbhi_score_official"], color="#2a78d6", ls=":", lw=1.3)
        ax.text(0, icbhi_ref["icbhi_score_official"] + 0.008, "ICBHI adult source",
                fontsize=7.5, color="#2a78d6")
    for xi, a in zip(xs, AGE_STRATA):
        ax.annotate(f"n={a['n_events']}", (xi, a["icbhi_score_official"]),
                    textcoords="offset points", xytext=(0, -13), ha="center", fontsize=7)
    ax.set_xticks(xs); ax.set_xticklabels([a["band"] for a in AGE_STRATA])
    ax.set_xlabel("age band (years)"); ax.set_ylabel("ICBHI challenge score")
    ax.set_title("Transfer score against patient age", fontsize=9.5)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(os.path.join(CFG["out_dir"], "M49_age_strata.png"), dpi=300, bbox_inches="tight")
    plt.show()

## Section 16 — Protocol-compliant results JSON

In [ ]:
# ============================================================
# CELL 19 — RESULTS JSON (Model_Training_Protocol.md section 4)
# ============================================================
head = ARMS["zeroshot_4class_broad"]
gap = (round(icbhi_ref["icbhi_score_official"] - head["icbhi_score_official"], 4)
       if ICBHI_VERIFIED else None)

results = {
    "meta": {
        "model_id": CFG["model_id"],
        "model_name": CFG["model_name"],
        "contributor": CFG["contributor"],
        "date_completed": time.strftime("%Y-%m-%d"),
        "is_augmented": False,
        "augmentation_method": "none (inference only; SpecAugment is training-time)",
        "notes": (
            "Zero-shot cross-dataset transfer. No training, no fine-tuning, no threshold "
            "fitting. The M22_v2 checkpoint is loaded frozen and scored on SPRSound 2022. "
            "The headline arm is zeroshot_4class_broad. prior_corrected_4class_broad uses "
            "the target class prior and is a diagnostic decomposition, not a zero-shot "
            "result. Comparison against the ICBHI source score is unpaired (different "
            "corpora, different patients), so bootstrap intervals are reported and no "
            "McNemar test is claimed."),
    },
    "config": {**{k: CFG[k] for k in (
        "sample_rate", "n_mels", "n_fft", "hop_length", "win_length", "f_min", "f_max",
        "duration_s", "n_frames", "arch", "batch_size", "seed", "num_classes")},
        "training": "none — inference only",
        "checkpoint_path": CKPT,
        "checkpoint_epoch": CKPT_EPOCH,
        "checkpoint_best_score": round(CKPT_SCORE, 4),
    },
    "environment": {
        "platform": PLATFORM,
        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "pytorch_version": torch.__version__,
        "python_version": sys.version.split()[0],
        "librosa_version": librosa.__version__,
    },
    "dataset_info": {
        "dataset": "SPRSound_2022",
        "source_dataset": "ICBHI_2017",
        "source_model_id": CFG["source_model_id"],
        "split_method": "external_corpus_zero_shot_no_split",
        "test_samples": int(len(spr_eval)),
        "test_patients": int(spr_eval.patient_id.nunique()),
        "test_recordings": int(spr_eval.stem.nunique()),
        "native_sample_rate_note": ("SPRSound is distributed at 8 kHz and resampled to 16 kHz; "
                                    "f_max is 2000 Hz, below the 4 kHz source Nyquist, so no "
                                    "band is fabricated"),
        "events_dropped_poor_quality": int(n_poor),
        "event_types_observed": {str(k): int(v)
                                 for k, v in spr.event_type.value_counts().items()},
        "event_types_unmapped": unmapped,
    },
    "efficiency": {
        "total_params": int(TOTAL_PARAMS),
        "trainable_params": 0,
        "training_time_total_s": 0,
        "inference_time_ms_per_sample": round(1000 * spr_infer_s / max(len(spr_eval), 1), 3),
        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    },
    "best_epoch": {
        "epoch": CKPT_EPOCH,
        "primary_metric": "icbhi_score_official",
        "primary_metric_value": head["icbhi_score_official"],
    },
    "best_metrics": {**{k: v for k, v in head.items() if k != "note"},
                     "icbhi_score_official_ci95": head["ci95"]},
    "verification": {
        "verified_against_source": bool(ICBHI_VERIFIED),
        "source_score_expected": CFG["source_score_icbhi"],
        "source_score_reproduced": (icbhi_ref["icbhi_score_official"] if ICBHI_VERIFIED else None),
        "source_score_delta": (round(icbhi_ref["icbhi_score_official"] - CFG["source_score_icbhi"], 4)
                               if ICBHI_VERIFIED else None),
        "source_reference_metrics": icbhi_ref,
        "source_binary_reference": icbhi_bin,
    },
    "transfer": {
        "arms": ARMS,
        "baselines_on_target": BASELINES,
        "generalization_gap_icbhi_minus_sprsound": gap,
        "calibration": CAL,
        "age_strata": AGE_STRATA,
        "target_class_prior": [round(float(p), 4) for p in prior_spr],
        "source_class_prior": [round(float(p), 4) for p in prior_src],
        "label_mapping_strict": STRICT_MAP,
        "label_mapping_broad": BROAD_MAP,
    },
    "ablation": {
        "ablation_group": "ood_generalization",
        "ablation_role": "variant",
        "baseline_model_id": CFG["source_model_id"],
        "variable_changed": "evaluation corpus: ICBHI 2017 -> SPRSound 2022 (weights frozen)",
        "variables_held_constant": [
            "checkpoint: M22_v2 best_model.pth",
            "architecture: mobilenet_v2 + ImageNet normalisation",
            "preprocessing: 128 mel / 16 kHz / 8 s / 50-2000 Hz / wrap-pad",
            "metric: icbhi_score_official",
            "seed: 42",
        ],
        "component_flags": {
            "has_sound_event_head": True, "has_disease_head": False,
            "has_cross_task_consistency": False, "has_cqkd_regularization": False,
            "has_openmax_rejection": False, "owl_stage": 0, "compression_clusters": None,
            "has_concept_bottleneck": False, "bottleneck_type": None, "concept_source": None,
            "has_leakage_measurement": False, "has_concept_intervention": False,
            "concept_space_ood": False, "fm_backbone": "none",
        },
        "loss_weights": {"sound_event_weight": 1.0, "disease_weight": None,
                         "consistency_weight": None},
    },
    "training_history": [],
}

def _json_safe(o):
    """NaN/Inf are not valid JSON and a downstream parser will choke on them."""
    if isinstance(o, dict):
        return {str(k): _json_safe(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_json_safe(v) for v in o]
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating, float)):
        f = float(o)
        return None if (math.isnan(f) or math.isinf(f)) else f
    if isinstance(o, (np.bool_,)):
        return bool(o)
    if isinstance(o, np.ndarray):
        return _json_safe(o.tolist())
    return o

out = os.path.join(CFG["out_dir"], "results_M49.json")
with open(out, "w") as fh:
    json.dump(_json_safe(results), fh, indent=2, allow_nan=False)
print(f"wrote {out}  ({os.path.getsize(out) / 1024:.1f} KB)")

## Section 17 — Summary

In [ ]:
# ============================================================
# CELL 20 — SUMMARY
# ============================================================
W = 74
print("=" * W)
print(f"  M49 — ICBHI 2017 -> SPRSound 2022, zero-shot, frozen MobileNetV2 + SpecAugment")
print("=" * W)

if ICBHI_VERIFIED:
    print(f"  checkpoint verified on ICBHI : {icbhi_ref['icbhi_score_official']:.4f} "
          f"(paper 0.5602, delta {icbhi_ref['icbhi_score_official'] - 0.5602:+.4f})")
else:
    print("  checkpoint NOT verified against ICBHI — every number below carries that caveat")
print("-" * W)
print(f"  {'arm':<34}{'score':>9}{'95% CI':>20}")
print("-" * W)
if ICBHI_VERIFIED:
    print(f"  {'ICBHI 4-class (source)':<34}{icbhi_ref['icbhi_score_official']:>9.4f}"
          f"{str(icbhi_ref['ci95']):>20}")
for k in ("zeroshot_4class_strict", "zeroshot_4class_broad",
          "zeroshot_binary_strict", "zeroshot_binary_broad",
          "prior_corrected_4class_broad"):
    print(f"  {k:<34}{ARMS[k]['icbhi_score_official']:>9.4f}{str(ARMS[k]['ci95']):>20}")
print("-" * W)
for k, v in BASELINES.items():
    print(f"  {'baseline: ' + k:<34}{v['icbhi_score_official']:>9.4f}")
print("=" * W)

if gap is not None:
    print(f"\n  generalization gap (ICBHI - SPRSound, 4-class broad): {gap:+.4f}")
lo, hi = ARMS["zeroshot_4class_broad"]["ci95"]
if lo > 0.5:
    print("  The transfer interval excludes chance: the representation carries something.")
else:
    print("  The transfer interval includes 0.5: on this corpus the frozen model is not")
    print("  shown to beat chance. State it that way — not 'it fails', not 'it works'.")

print("\n  Files in " + CFG["out_dir"] + ":")
for f in sorted(os.listdir(CFG["out_dir"])):
    print(f"    {f}  ({os.path.getsize(os.path.join(CFG['out_dir'], f)) / 1024:.0f} KB)")

print("""
  How to read this
  ----------------
  The point is not a high number. A frozen adult-trained model scored on paediatric audio
  from a different stethoscope is expected to drop, and the honest contribution is the
  size of the drop with an interval around it, next to the trivial baselines on the same
  corpus. The paper's own argument is that evaluation protocol is worth more than
  architecture, and this measurement is that argument applied to itself.

  Report the broad 4-class arm as the headline, the strict arm beside it as the label
  sensitivity check, the binary arm as the fair-transfer test, and the prior-corrected
  arm only as decomposition.
""")

In [ ]:
# ============================================================
# CELL 21 — HANDOFF DOWNLOADS (protocol section 11.D)
# ============================================================
import shutil
from IPython.display import display, FileLink

files = sorted(glob.glob(os.path.join(CFG["out_dir"], "*")))
print("=" * 60)
print("M49 OUTPUTS")
print("=" * 60)
for f in files:
    print(f"Ready: {os.path.basename(f):<34} ({os.path.getsize(f) / (1024 * 1024):.2f} MB)")

zip_path = shutil.make_archive(os.path.join(WORK, "M49_handoff"), "zip", CFG["out_dir"])
print(f"\nbundle: {zip_path} ({os.path.getsize(zip_path) / (1024 * 1024):.2f} MB)")

if PLATFORM == "Colab":
    try:
        from google.colab import drive, files as colab_files
        dest = "/content/drive/MyDrive/OWMTL/M49"
        if os.path.isdir("/content/drive/MyDrive"):
            os.makedirs(dest, exist_ok=True)
            for f in files:
                shutil.copy2(f, dest)
            print(f"copied results to {dest}")
        colab_files.download(zip_path)
    except Exception as e:
        print(f"(Drive copy / download skipped: {e})")
else:
    for f in files:
        display(FileLink(f))
    display(FileLink(zip_path))